# Semantic Graph Builder v2 — Interactive Notebook

Run each stage individually and inspect intermediate results.

In [ ]:
import sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

_root = Path().resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_here = Path().resolve()
if str(_here) in sys.path:
    sys.path.remove(str(_here))

from llm_v2.config_schema import load_config
config = load_config('config.yaml')
print(config.model_dump_json(indent=2))

In [ ]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

In [ ]:
from llm_v2.utils.io import load_text
from pathlib import Path

text = load_text(config.paths.input_text)
print(f'Text length: {len(text)} chars')
print(text[:500])

## [0] Preprocessing

In [ ]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

## [1] Coreference Resolution

In [ ]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=Path('.')
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

## [1.5] Chunking

In [ ]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

## [2] Triplet Extraction

In [ ]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=Path('.'))
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

## [3] Normalization

In [ ]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

## [4] Deduplication

In [ ]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

## [5] Graph Assembly (raw)

In [ ]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

## [6] Clustering

In [ ]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt
from pathlib import Path

# load cluster naming prompt
naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print(f"Multi-method clustering:")
    for method_name, mr in multi.methods.items():
        print(f"  {method_name}: {len(mr.param_labels)} variants")
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f"    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges")
    agg = multi.methods["agglomerative"]
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods["agglomerative"]
    print(f"Multi-threshold: {len(agg.param_labels)} levels")
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f"  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges")
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

## Save outputs

In [ ]:
from llm_v2.utils.io import save_json, save_text

out = Path(config.paths.output_dir)
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

## Benchmark

In [ ]:
from llm_v2.benchmark import (
    evaluate_graph,
    load_clustered_graph,
    print_metrics,
    show_node_alignments,
    show_edge_alignments,
)

gt_graph_path = Path('../benchmark/final_bench/graph_clustered.json')
gt_text_path  = Path('../benchmark/final_bench/formated_fragment2.md')

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

source_text = resolved_text if resolved_text.strip() else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

In [ ]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    node_window=300,
    edge_window=400,
)

print_metrics(metrics)
metrics.summary()